# Notebook 3: Visualisation & Follow-up Analysis
**H9DISS1 Data Intensive Scalable Systems | NCI MSc Data Analytics 2025/26**

---

Reads Gold Delta tables from the Lakehouse and produces all five
publication-quality figures for the IEEE report.

| Figure | Content | Answers |
|--------|---------|--------|
| Fig 1 | SRI vs IA activation rate scatter | RQ1 correlation |
| Fig 2 | Multi-hazard portfolio + IA vs PA bar | RQ2 portfolio structure |
| Fig 3 | Declaration breadth vs IA activations bubble | RQ2 insurance gap |
| Fig 4 | Temporal trend — county rows & aid composition | RQ3 post-2010 shift |
| Fig 5 | IA rate heatmap by incident type × state | RQ2/RQ3 cross-hazard |

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")   # non-interactive backend for Fabric
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings("ignore")

# Read Gold tables from Lakehouse as pandas DataFrames
a1     = spark.table("gold_analysis1_eq_aid_correlation").toPandas()
a2     = spark.table("gold_analysis2_multihazard_profile").toPandas()
a2_eq  = spark.table("gold_analysis2_eq_state_profile").toPandas()
a3_nat = spark.table("gold_analysis3_national_trend").toPandas()
a3_st  = spark.table("gold_analysis3_annual_by_state").toPandas()

# ── Style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         11,
    "axes.titlesize":    13,
    "axes.labelsize":    11,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
    "savefig.dpi":       200,
    "savefig.bbox":      "tight",
    "savefig.facecolor": "white",
})
C = {
    "primary":   "#1B4F72",
    "secondary": "#2E86AB",
    "accent":    "#E84855",
    "warn":      "#F4A261",
    "ok":        "#2DC653",
    "neutral":   "#95A5A6",
}

# Output path inside Lakehouse Files (Bronze → Gold figures)
FIG_PATH = "Files/earthquake_insurance/figures/"

# Helper: save figure to Lakehouse AND display inline in Fabric
def save_fig(fig, filename):
    local = f"/tmp/{filename}"
    fig.savefig(local)
    # Upload to Lakehouse Files
    mssparkutils.fs.cp(f"file:{local}", f"{FIG_PATH}{filename}", overwrite=True)
    plt.show()    # renders inline in Fabric notebook
    plt.close(fig)
    print(f"  Saved: {FIG_PATH}{filename}")

print("Gold tables loaded. Generating figures...")
print(f"  A1: {len(a1)} states | A2: {len(a2)} combos | A3 national: {len(a3_nat)} years")

## Figure 1: Seismic Risk Index vs IA Activation Rate (RQ1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

x     = a1["sri"].fillna(0)
y     = a1["ia_rate"].fillna(0)
sizes = (a1["county_declarations"] / a1["county_declarations"].max() * 500).clip(40, 500)

sc = ax.scatter(x, y, s=sizes, c=a1["pa_rate"].fillna(0), cmap="YlOrRd",
                edgecolors=C["primary"], linewidths=0.7, alpha=0.85, zorder=3)

# Regression line
if len(a1) > 2:
    slope, intercept, r, p, _ = stats.linregress(x, y)
    xfit = np.linspace(x.min(), x.max(), 100)
    ax.plot(xfit, slope*xfit + intercept, "--", color=C["accent"],
            linewidth=2, label=f"Regression  r = {r:.3f}  (p = {p:.3f})")

# State labels
for _, row in a1.iterrows():
    ax.annotate(row["state"],
                xy=(row["sri"], row["ia_rate"]),
                xytext=(6, 4), textcoords="offset points",
                fontsize=9, color=C["primary"], fontweight="bold")

cbar = plt.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("PA Activation Rate", fontsize=10)

ax.set_xlabel("FEMA Seismic Risk Index (SRI) — composite from real declaration data")
ax.set_ylabel("Individual Assistance (IA) Activation Rate")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_title("Figure 1: Seismic Risk Index vs Individual Assistance Activation Rate\n"
             "(FEMA Disaster Declarations 1953–2026, real data)",
             fontweight="bold", pad=10)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3, zorder=0)
ax.text(0.03, 0.97, f"Pearson r = {r:.3f}",
        transform=ax.transAxes, fontsize=10, va="top",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                  edgecolor="grey", alpha=0.8))
save_fig(fig, "fig1_sri_ia_scatter.png")

## Figure 2: Multi-Hazard Profile + IA vs PA Rates (RQ2)

In [ ]:
EQ_STATES  = ["CA","WA","AK","OR","UT","HI","ID","NV","PR","GU","AS"]
TOP_TYPES  = ["Earthquake","Fire","Flood","Severe Storm","Hurricane",
              "Biological","Typhoon","Mud/Landslide","Drought"]
PALETTE    = [C["accent"],C["warn"],"#4ECDC4","#45B7D1","#96CEB4",
              "#FFEAA7","#DDA0DD","#98D8C8",C["neutral"]]

eq_sorted = a2_eq.sort_values("county_rows", ascending=False)
states_ord = eq_sorted["state"].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: stacked bar — disaster portfolio
ax = axes[0]
x  = np.arange(len(states_ord))
bottom = np.zeros(len(states_ord))
for t, col_c in zip(TOP_TYPES, PALETTE):
    vals = []
    for st in states_ord:
        r = a2[(a2["state"]==st) & (a2["incidentType"]==t)]["declarations"]
        vals.append(int(r.sum()) if len(r) else 0)
    ax.bar(x, vals, bottom=bottom, label=t, color=col_c, width=0.6, zorder=3)
    bottom += np.array(vals)
ax.set_xticks(x)
ax.set_xticklabels(states_ord, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Unique Disaster Declarations")
ax.set_title("(a) Multi-Hazard Disaster Portfolio\n(Seismically Active States)",
             fontweight="bold")
ax.legend(fontsize=7, ncol=2, loc="upper right")
ax.grid(axis="y", alpha=0.3, zorder=0)

# Right: IA vs PA activation rates for earthquake declarations
ax = axes[1]
eq_s = a2_eq.sort_values("ia_rate", ascending=False)
x2   = np.arange(len(eq_s))
ax.bar(x2-0.18, eq_s["ia_rate"], width=0.35,
       color=C["secondary"], label="IA Rate (Individual Assistance)", zorder=3)
ax.bar(x2+0.18, eq_s["pa_rate"], width=0.35,
       color=C["accent"],    label="PA Rate (Public Assistance)",     zorder=3)
ax.set_xticks(x2)
ax.set_xticklabels(eq_s["state"], rotation=35, ha="right", fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_ylabel("Program Activation Rate (% of EQ declarations)")
ax.set_title("(b) IA vs PA Activation Rates\nfor Earthquake Declarations by State",
             fontweight="bold")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3, zorder=0)

fig.suptitle("Figure 2: Multi-Hazard Profile of Seismically Active States (Real FEMA Data)",
             fontweight="bold", fontsize=13, y=1.01)
fig.tight_layout()
save_fig(fig, "fig2_multihazard_profile.png")

## Figure 3: Declaration Breadth vs IA Activations (Insurance Gap)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

x     = a1["county_declarations"]
y     = a1["ia_activations"]
sizes = (a1["total_declarations"] / a1["total_declarations"].max() * 500).clip(40, 500)

sc = ax.scatter(x, y, s=sizes, c=a1["sri"].fillna(0), cmap="RdYlGn_r",
                edgecolors="white", linewidths=0.8, alpha=0.85, zorder=3)

for _, row in a1.iterrows():
    ax.annotate(row["state"],
                xy=(row["county_declarations"], row["ia_activations"]),
                xytext=(7, 3), textcoords="offset points",
                fontsize=9.5, color=C["primary"], fontweight="bold")

# Reference diagonal: if IA activated for every county declaration
xmax = x.max() * 1.1
ax.plot([0, xmax], [0, xmax], "--", color=C["neutral"], linewidth=1.2,
        label="IA = 100% of county declarations (reference)")

# Annotate Puerto Rico finding
pr = a1[a1["state"]=="PR"]
if len(pr):
    ax.annotate(
        f"PR: {int(pr['county_declarations'].values[0])} county decl.\n"
        f"0 IA activations (0%)",
        xy=(pr["county_declarations"].values[0], pr["ia_activations"].values[0]),
        xytext=(55, 18), textcoords="offset points",
        fontsize=8.5, color=C["accent"], fontweight="bold",
        arrowprops=dict(arrowstyle="->", color=C["accent"], lw=1.5)
    )

cbar = plt.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("Seismic Risk Index (SRI)", fontsize=10)
ax.set_xlabel("Total Earthquake-Affected County Declarations (FEMA)")
ax.set_ylabel("Individual Assistance (IA) Activations")
ax.set_title("Figure 3: Declaration Breadth vs IA Activations by State\n"
             "(Bubble size = unique disaster events; states below diagonal = insurance gap)",
             fontweight="bold", pad=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.25, zorder=0)
save_fig(fig, "fig3_declaration_breadth.png")

## Figure 4: Temporal Trends — Declaration Breadth & Aid Composition (RQ3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

yr_data = a3_nat.dropna(subset=["total_county_rows"]).sort_values("year")

# Left: county declaration count by year
ax = axes[0]
ax.fill_between(yr_data["year"], yr_data["total_county_rows"],
                alpha=0.2, color=C["secondary"])
ax.plot(yr_data["year"], yr_data["total_county_rows"],
        marker="o", color=C["secondary"], linewidth=2.2, markersize=6)
# Annotate peak years
for _, row in yr_data.nlargest(3, "total_county_rows").iterrows():
    ax.annotate(str(int(row["year"])),
                xy=(row["year"], row["total_county_rows"]),
                xytext=(0, 10), textcoords="offset points",
                ha="center", fontsize=9, color=C["accent"], fontweight="bold")
ax.axvline(2010, color=C["accent"], linewidth=1.2, linestyle=":",
           label="2010 threshold (post-2010: IA rate → 0%)")
ax.set_xlabel("Year")
ax.set_ylabel("FEMA EQ-Affected County Declarations")
ax.set_title("(a) Annual Earthquake Declaration Breadth\n(2000–2024)", fontweight="bold")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Right: IA vs PA stacked per year
ax = axes[1]
ax.bar(yr_data["year"], yr_data["total_ia"],
       label="IA (Individual Assistance — residential)",
       color=C["secondary"], zorder=3)
ax.bar(yr_data["year"], yr_data["total_pa"],
       bottom=yr_data["total_ia"],
       label="PA (Public Assistance — infrastructure)",
       color=C["accent"], zorder=3)
ax.axvline(2010, color="black", linewidth=1.2, linestyle=":",
           label="2010 — IA drops to 0 permanently")
ax.set_xlabel("Year")
ax.set_ylabel("Federal Aid Program Activations")
ax.set_title("(b) IA vs PA Program Activations by Year\n(post-2002: IA = 0 in every year)",
             fontweight="bold")
ax.legend(fontsize=8.5)
ax.grid(axis="y", alpha=0.3, zorder=0)

fig.suptitle("Figure 4: Temporal Trends in FEMA Earthquake Disaster Response (2000–2024)",
             fontweight="bold", fontsize=13, y=1.01)
fig.tight_layout()
save_fig(fig, "fig4_temporal_trend.png")

## Figure 5: IA Rate Heatmap by Incident Type × State

In [ ]:
EQ_STATES_HEAT = ["CA","WA","AK","OR","HI","ID","NV","PR"]
TOP_TYPES_HEAT = ["Earthquake","Fire","Flood","Severe Storm",
                  "Hurricane","Biological","Typhoon","Drought"]

sub = a2[a2["state"].isin(EQ_STATES_HEAT) &
         a2["incidentType"].isin(TOP_TYPES_HEAT)]
pivot = sub.pivot_table(
    index="incidentType", columns="state",
    values="ia_rate", aggfunc="mean"
).fillna(0)
pivot = pivot.reindex(
    index=[t for t in TOP_TYPES_HEAT  if t in pivot.index],
    columns=[c for c in EQ_STATES_HEAT if c in pivot.columns]
)

fig, ax = plt.subplots(figsize=(11, 5.5))
cmap = LinearSegmentedColormap.from_list(
    "ia", ["#EBF5FB","#2E86AB","#1B4F72","#0A1F30"]
)
sns.heatmap(pivot, ax=ax, cmap=cmap, annot=True, fmt=".2f",
            linewidths=0.5, linecolor="white", vmin=0, vmax=1,
            cbar_kws={"label": "IA Activation Rate (0–1)"})

ax.set_title(
    "Figure 5: Individual Assistance (IA) Activation Rate by Incident Type × State\n"
    "(Higher = more residential earthquake insurance exposure; "
    "orange border = earthquake row)",
    fontweight="bold", pad=12
)
ax.set_xlabel("State")
ax.set_ylabel("Incident Type")

# Highlight earthquake row
idx_list = list(pivot.index)
if "Earthquake" in idx_list:
    eq_row = idx_list.index("Earthquake")
    ax.add_patch(plt.Rectangle(
        (0, eq_row), pivot.shape[1], 1,
        fill=False, edgecolor="#E84855", lw=2.5, zorder=5
    ))

save_fig(fig, "fig5_ia_heatmap.png")

## Summary: All Figures Saved

In [ ]:
print("="*60)
print(" NOTEBOOK 3 — VISUALISATION COMPLETE")
print("="*60)
figures = [
    ("fig1_sri_ia_scatter.png",    "RQ1 — SRI vs IA activation rate scatter"),
    ("fig2_multihazard_profile.png","RQ2 — Multi-hazard portfolio + IA/PA rates"),
    ("fig3_declaration_breadth.png","RQ2 — Declaration breadth vs IA activations"),
    ("fig4_temporal_trend.png",     "RQ3 — Temporal trend + post-2010 IA gap"),
    ("fig5_ia_heatmap.png",         "RQ2/3 — IA rate heatmap incident × state"),
]
for fn, desc in figures:
    print(f"  Files/earthquake_insurance/figures/{fn}")
    print(f"  → {desc}")
    print()
print(" All 6 Gold tables + 5 figures written to Lakehouse.")
print(" Proceed to Notebook 4: Data Warehouse & Power BI.")
print("="*60)